# Modell zur Ausgabenkategorisierung

Das Modell zur Ausgabenkategorisierung verfolgt einen hybriden Ansatz, der regelbasierte Klassifikation, einen persistenten Speicher sowie ein lokal ausgeführtes Large Language Model (LLM) kombiniert. Ziel ist es, Banktransaktionen möglichst präzise zu kategorisieren und gleichzeitig die Anzahl der LLM-Abfragen zu minimieren.

Der Klassifikationsprozess besteht aus den folgenden Schritten:

* **Vorverarbeitung der Transaktion:** Der Zahlungsempfänger (Payee) wird normalisiert, indem Groß- und Kleinschreibung vereinheitlicht sowie Umlaute, Zahlen, Satzzeichen, URLs und Ortsnamen entfernt werden. Dadurch können unterschiedliche Schreibweisen desselben Händlers zuverlässig erkannt werden.
* **Speicherabfrage:** Der normalisierte Händlername wird mit einer lokalen SQLite-Datenbank verglichen, die bereits kategorisierte Händler enthält. Bei einer Übereinstimmung wird die gespeicherte Kategorie direkt übernommen.
* **Regelbasierte Klassifikation:** Ist kein Eintrag vorhanden, werden vordefinierte Schlüsselwortregeln für häufig vorkommende Händler (z. B. Supermärkte, Streaming-Dienste oder Verkehrsunternehmen) angewendet.
* **LLM als Fallback:** Kann die Transaktion weder durch den Speicher noch durch die Regeln kategorisiert werden, wird ein lokal über Ollama ausgeführtes LLM verwendet (qwen2.5:3b). Dieses ordnet die Transaktion einer der vorgegebenen Ausgabenkategorien zu.
* **Aktualisierung des Speichers:** Neu klassifizierte Händler werden in der SQLite-Datenbank gespeichert, sodass zukünftige Transaktionen desselben Händlers direkt aus dem Speicher kategorisiert werden können.

Zusätzlich unterstützt das System manuelle Korrekturen. Wird eine Kategorie durch den Benutzer angepasst, wird der entsprechende Eintrag in der Datenbank aktualisiert. Dadurch werden zukünftige Transaktionen desselben Händlers automatisch mit der korrigierten Kategorie versehen. Auf diese Weise passt sich das System kontinuierlich an das individuelle Ausgabeverhalten des Nutzers an, während sämtliche Finanzdaten lokal auf dem Gerät verbleiben.


In [1]:
# ollama pull qwen2.5:3b

import sqlite3
import json
import re
from pathlib import Path

import pandas as pd
import ollama
import geonamescache
import unicodedata

# import DKB parser temporarily to get real data
import sys
sys.path.append("../../")
from backend.parsers import DKB

gc = geonamescache.GeonamesCache()

# List of city names for filtering out from payee strings
city_names = {
    "".join(
        c for c in unicodedata.normalize("NFKD", city["name"].lower())
        if not unicodedata.combining(c)
    )
    for city in gc.get_cities().values()
}

# Function to clean and normalize (payee) strings
def clean_payee(text: str) -> str:
    """
    Normalizes a payee string by:
      - converting to lowercase
      - removing accents/umlauts
      - removing URLs
      - removing standalone numbers
      - removing special characters and punctuation
      - removing city names
      - collapsing multiple spaces
    """
    if pd.isna(text):
        return ""

    # Lowercase and strip whitespace
    text = str(text).lower().strip()

    # Remove accents (ä -> a, é -> e, ...)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))

    # Remove URLs/domains
    text = re.sub(r"\.com\b|\.co\.uk\b|www\.", " ", text)

    # Remove standalone numbers
    text = re.sub(r"\b\d+\b", " ", text)

    # Remove special characters
    text = re.sub(r"[*#]", " ", text)

    # Remove punctuation (keep letters, digits and spaces)
    text = re.sub(r"[^\w\s]", " ", text)

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Remove city names
    words = text.split()
    text = " ".join(word for word in words if word not in city_names)

    return text


# ----------------------------
# SQLite memory
# ----------------------------
DB_PATH = Path("expense_memory.sqlite")

CATEGORIES = [
    "Lebensmittel",
    "Transport",
    "Miete",
    "Nebenkosten",
    "Entertainment",
    "Shopping",
    "Gesundheit",
    "Einkommen",
    "Sonstiges",
    "Hobby"
]

CATEGORIES = ['8. fixed costs',      
              '3. mobility',     
              '1. groceries',
              '4. family',    
              '6. healthcare', 
              '2. lifestyle&fun',
              '5. education',         
              '7. abos'
              ]

def init_database():
    conn = sqlite3.connect(DB_PATH)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS merchant_categories (
            merchant TEXT PRIMARY KEY,
            category TEXT NOT NULL,
            confidence REAL DEFAULT 1.0
        )
    """)

    conn.commit()
    return conn


def lookup_memory(conn, description):
    """
    Look for previously classified merchants.
    """

    text = description.upper()

    rows = conn.execute(
        "SELECT merchant, category, confidence FROM merchant_categories"
    ).fetchall()

    for merchant, category, confidence in rows:
        if merchant in text:
            return {
                "category": category,
                "confidence": confidence,
                "source": "memory"
            }

    return None


def save_memory(conn, merchant, category, confidence):
    conn.execute(
        """
        INSERT OR REPLACE INTO merchant_categories
        (merchant, category, confidence)
        VALUES (?, ?, ?)
        """,
        (merchant.upper(), category, confidence)
    )

    conn.commit()

def correct_category(conn, merchant, new_category):
    """
    Manually corrects a merchant category.
    Future transactions with this merchant will use the corrected category.
    """

    if new_category not in CATEGORIES:
        raise ValueError(
            f"Unknown category: {new_category}"
        )

    conn.execute(
        """
        INSERT OR REPLACE INTO merchant_categories
        (merchant, category, confidence)
        VALUES (?, ?, ?)
        """,
        (
            merchant.upper(),
            new_category,
            1.0
        )
    )

    conn.commit()


# ----------------------------
# Rule engine
# ----------------------------

# TODO: Expand categories and check if it really has to be hard coded
RULES = {
    "1. groceries": [
        "rewe",
        "lidl",
        "aldi",
        "edeka",
        "netto",
        "penny",
        "markant"
    ],
    "3. mobility": [
        "db ",
        "deutsche bahn",
        "flix",
        "uber",
        "shell",
        "aral",
    ],
    "7. abos": [
        "netflix",
        "spotify",
        "steam",
        "disney",
    ],
    "8. fixed costs": [
        "telekom",
        "vodafone",
        "energie",
        "strom",
    ],
    "6. healthcare": [
        "apotheke",
        "pharmacy",
        "arzt",
    ],
    "2. lifestyle&fun": [
        "wolle"
    ]
}


def apply_rules(description):

    if pd.isna(description):
        return None

    text = description.lower()

    for category, keywords in RULES.items():

        for keyword in keywords:

            if keyword.lower() in text:
                return {
                    "category": category,
                    "confidence": 0.95,
                    "source": "rules"
                }

    return None


# ----------------------------
# LLM fallback
# ----------------------------

def classify_with_llm(description):

    prompt = f"""
    You are a transaction classification engine for personal finance.

    Your task is to classify ONE German bank transaction into exactly ONE category.

    The transaction description comes from a German bank export and may contain a lot of irrelevant banking information.

    Your classification process:

    1. Ignore banking metadata:
    - payment methods (Kartenzahlung, Lastschrift, Überweisung, SEPA)
    - payment processors (PAYPAL, VR PAYMENT, SUMUP, STRIPE)
    - transaction IDs, reference numbers, terminal IDs
    - dates, booking codes, store numbers
    - locations unless they help identify the merchant

    2. Identify the actual merchant or recipient.

    3. Determine what type of expense this represents.

    4. Select the most appropriate category.

    Available categories:

    - "8. fixed costs"
    Housing-related recurring costs and essential bills:
    rent, insurance, electricity, internet, phone contracts, subscriptions that are not entertainment

    - "3. mobility"
    Transportation:
    trains, buses, public transport, fuel, parking, taxis, car-related costs

    - "1. groceries"
    Food and everyday household supplies:
    supermarkets, grocery stores, bakeries, drugstores for household goods

    - "4. family"
    Expenses related to children or family:
    childcare, kindergarten, school meals, family activities

    - "6. healthcare"
    Medical expenses:
    doctors, pharmacies, medication, health services, therapy

    - "2. lifestyle&fun"
    Leisure and discretionary spending:
    restaurants, cafes, bars, cinema, games, hobbies, sports activities, entertainment

    - "5. education"
    Learning and professional development:
    courses, books for learning, training, education platforms

    - "7. abos"
    Recurring digital subscriptions:
    Netflix, Spotify, Amazon Prime, software subscriptions, streaming services

    Important rules:

    - Classify the merchant, NOT the payment provider.
    Example:
    "PAYPAL *SPOTIFY" -> Spotify -> "7. abos"
    NOT PayPal

    - Be consistent:
    The same merchant must always receive the same category.

    - If a merchant clearly fits a category, choose it confidently.

    - Use "8. fixed costs" only for recurring necessary expenses.
    Do NOT put entertainment subscriptions there.

    - Use "7. abos" only for subscriptions.
    A one-time purchase from Amazon is not an abo.

    - If uncertain, choose the closest category and lower the confidence score.

    Examples:

    "REWE SAGT DANKE"
    => groceries

    "ALDI SUED FIL.017 KIEL"
    => groceries

    "PAYPAL *SPOTIFY"
    => abos

    "NETFLIX.COM"
    => abos

    "DEUTSCHE BAHN AG"
    => mobility

    "SHELL STATION 1234"
    => mobility

    "TK MAXX"
    => lifestyle&fun

    "APOTHEKE AM MARKT"
    => healthcare

    "VODAFONE GMBH"
    => fixed costs

    "UDACITY COURSE"
    => education


    Transaction:

    {description}

    Return ONLY valid JSON. No explanation.

    Format:

    {{
        "merchant": "identified merchant name",
        "category": "one category from the list",
        "confidence": 0.0
    }}
    """

    response = ollama.chat(
        model="qwen2.5:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        format="json"   # important
    )

    result = json.loads(
        response["message"]["content"]
    )

    return {
        "category": result["category"],
        "confidence": float(result["confidence"]),
        "source": "llm"
    }


# ----------------------------
# Merchant extraction
# ----------------------------

def extract_merchant(description):
    """
    Crude normalization.
    Improve this over time.
    """

    text = description.upper()

    text = re.sub(
        r"\d+",
        "",
        text
    )

    text = re.sub(
        r"[^A-ZÄÖÜ ]",
        " ",
        text
    )

    words = text.split()

    return " ".join(words[:2])


# Function to manually correct a transaction's category and save it to the database
def correct_transaction(df, index, new_category):

    conn = init_database()

    description = df.loc[index, "classifier"]

    merchant = clean_payee(description)

    correct_category(
        conn,
        merchant,
        new_category
    )

    conn.close()

In [2]:
# ----------------------------
# Main classifier
# ----------------------------

def classify_expenses(df):

    conn = init_database()

    results = []

    for _, row in df.iterrows():

        description = row["classifier"]


        result = (
            lookup_memory(conn, description)
            or
            apply_rules(description)
            or
            classify_with_llm(clean_payee(description))
        )


        merchant = clean_payee(description)


        # Remember LLM/rule decisions
        save_memory(
            conn,
            merchant,
            result["category"],
            result["confidence"]
        )


        results.append(result)


    output = df.copy()

    output["category"] = [
        r["category"]
        for r in results
    ]

    output["confidence"] = [
        r["confidence"]
        for r in results
    ]

    output["classification_source"] = [
        r["source"]
        for r in results
    ]

    conn.close()

    return output

## Test der Pipeline mit kategorisierten Daten
Die Daten wurden synthetisch erstellt und kategorisiert.

In [3]:
import pandas as pd
# from expense_classifier import classify_expenses

transactions = DKB.load("../../data/dkb_synth_large.csv") # synthetic dataset
transactions["classifier"] = transactions["payee"] + " " + transactions["description"]
transactions["classifier"] = transactions["classifier"].apply(clean_payee)

result = classify_expenses(transactions[["classifier"]].head(20))

print(result)


                                           classifier          category  \
0                            1und1 dsl rechnung kd nr    8. fixed costs   
1   paypal europe s a r l et cie s c a pp pp city ...          mobility   
2                   deutsche glasfaser rechnung kd nr    8. fixed costs   
3   spargelhof klein spargelhof sagt danke 16t22 k...    8. fixed costs   
4                   rossmann babywelt lastschrift ref         4. family   
5                          wilhelm tel rechnung kd nr    8. fixed costs   
6                                 kvb lastschrift ref    8. fixed costs   
7   paypal europe s a r l et cie s c a pp pp taxi ...       3. mobility   
8   paypal europe s a r l et cie s c a pp pp apoth...     6. healthcare   
9                            hotel hotel 14t16 kfn vj    8. fixed costs   
10  paypal europe s a r l et cie s c a pp pp hornb...    8. fixed costs   
11  paypal europe s a r l et cie s c a pp pp bike ...  2. lifestyle&fun   
12         book depositor

## Vergleich der Vorhergesagten Kategorien mit den Tatsächlichen

In [5]:
comparison = result.merge(
    transactions[["classifier", "category"]],
    on="classifier",
    suffixes=("_predicted", "_true")
)

comparison["correct"] = (
    comparison["category_predicted"] ==
    comparison["category_true"]
)

accuracy = comparison["correct"].mean()

print(f"Accuracy: {accuracy:.2%}")

correct = comparison["correct"].sum()
total = len(comparison)

print(f"Correct: {correct}/{total}")
print(f"Wrong: {total-correct}/{total}")


Accuracy: 57.75%
Correct: 41/71
Wrong: 30/71


In [13]:
correct_transaction(
    transactions,
    0,
    "Abrechnung"
)

correct_transaction(
    transactions,
    15,
    "Kleidung"
)